# 🐾 Deep Learning Background Removal (ResNet-34 U-Net)
### End-to-End Kaggle Training & Inference Pipeline

This notebook runs the complete background removal pipeline on Kaggle:
1. **Repository Clone & Setup** (pulls project code from GitHub)
2. **Environment & GPU Verification**
3. **Data & Asset Pre-checks** (handles Kaggle read-only paths and network timeouts)
4. **Step 1: Sanity Dry Run (2 Epochs)** to verify forward/backward passes and checkpointing
5. **Step 2: Full Production Training (100 Epochs)** with early stopping and CosineAnnealingLR
6. **Step 3: Training Curves & Metrics Visualization**
7. **Step 4: Inference & Qualitative Visual Comparisons** (Transparent PNG, Alpha Matte, Studio Background)
8. **Step 5: Bundle & Download Final Artifacts**

> **Kaggle Setting Reminder**: Ensure **Internet: ON** and **GPU: T4 x2 (or P100)** are selected in the right sidebar under *Settings*.

## 1. Clone Repository & Working Directory Setup
Clones the codebase from GitHub into `/kaggle/working` and navigates into the repository root.

In [ ]:
import os
import sys

REPO_NAME = "bg-removal-with-deep-learning"
REPO_URL = "https://github.com/Razamindset/bg-removal-with-deep-learning.git"

# Clone repository if we are in /kaggle/working and repo is not yet present
if not os.path.isdir("src"):
    if not os.path.isdir(REPO_NAME):
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL}
    if os.path.isdir(REPO_NAME):
        os.chdir(REPO_NAME)

# Add repository root to Python path
repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Current Working Directory: {os.getcwd()}")
print("Repository contents:", [f for f in os.listdir(".") if not f.startswith(".")])

## 2. System & GPU Environment Verification

In [ ]:
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"GPU Count: {device_count}")
    for i in range(device_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: No GPU detected! Please go to Kaggle notebook Settings -> Accelerator and choose GPU T4 x2 or P100.")

## 3. Dependencies & Pretrained Asset Pre-checks
Install project requirements and pre-cache the ResNet-34 ImageNet weights to eliminate mid-training download timeouts.

In [ ]:
!pip install -q -r requirements.txt

import torchvision.models as models
print("Pre-loading ResNet-34 pretrained weights into cache...")
_ = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
print("ResNet-34 weights successfully cached!")

## 4. Dataset Pre-check & Verification
Checks whether Oxford-IIIT Pet is attached as Kaggle input `/kaggle/input/` or needs downloading to `/kaggle/working/data`. Resolves read-only paths automatically.

In [ ]:
from src.dataset.pets import get_dataloaders, resolve_dataset_root

data_root, needs_download = resolve_dataset_root("/kaggle/working/data")
print(f"Resolved dataset path: {data_root}")
print(f"Needs download: {needs_download}")

# Test loading one sample batch
train_loader, val_loader, test_loader = get_dataloaders(
    root=data_root,
    image_size=256,
    batch_size=8,
    num_workers=2
)

images, masks, valids = next(iter(train_loader))
print(f"Images shape: {images.shape} (min: {images.min():.2f}, max: {images.max():.2f})")
print(f"Masks shape:  {masks.shape} (unique: {torch.unique(masks).tolist()})")
print(f"Dataset: {len(train_loader.dataset)} train, {len(val_loader.dataset)} val, {len(test_loader.dataset)} test samples")

## 5. Step 1: Sanity Check (2-Epoch Dry Run)
Runs a quick 2-epoch test using `configs/smoke_test.yaml` to ensure:
- Mixed Precision (AMP) backward pass executes without NaN
- Encoder backbone freezes properly on epoch 1
- Checkpoints (`best.pth`, `last.pth`) are saved properly
- Validation metrics compute without error

In [ ]:
print("Starting 2-Epoch Sanity Check...")
!python -m src.train --config configs/smoke_test.yaml
print("Sanity Check complete!")

## 6. Step 2: Full Production Training (100 Epochs)
Now run the full training using `configs/pets.yaml`:
- **Backbone Freezing**: Pretrained ResNet-34 is frozen for the first 3 epochs to stabilize the decoder
- **Learning Rate Scheduler**: Cosine Annealing scheduler smoothly decays LR
- **Gradient Clipping**: Norm clipped at 1.0 to prevent gradient spikes
- **Early Stopping**: Monitored on `val_dice` with 10 epochs patience

In [ ]:
print("Starting Production Training...")
!python -m src.train --config configs/pets.yaml

## 7. Step 3: Training Curves & Metrics Visualization
Generate publication-quality training graphs for Loss, Validation Dice/IoU, Learning Rates, and per-epoch training time.

In [ ]:
import matplotlib.pyplot as plt
from src.plot_history import plot_training_history

history_csv = "/kaggle/working/outputs/history.csv"
plot_output = "/kaggle/working/outputs/training_curves.png"

fig = plot_training_history(history_csv, plot_output, show=False)
plt.show()

## 8. Step 4: Inference & Qualitative Visual Comparisons
Run `src/predict.py` on test samples to extract transparent PNGs and replace backgrounds with solid colors.

In [ ]:
import glob
from PIL import Image

# Select 3 test images
test_images = sorted(glob.glob(f"{data_root}/oxford-iiit-pet/images/*.jpg"))[:3]
print("Sample images for testing:", test_images)

weights_path = "/kaggle/working/outputs/best.pth"
pred_output_dir = "/kaggle/working/outputs/predictions"

for img_path in test_images:
    # 1. Generate transparent PNG and alpha mask
    !python -m src.predict --weights {weights_path} --input {img_path} --output {pred_output_dir} --save-mask
    # 2. Generate solid white studio background replacement
    !python -m src.predict --weights {weights_path} --input {img_path} --output {pred_output_dir} --bg-color 255 255 255

### Qualitative Side-by-Side Display

In [ ]:
import numpy as np

fig, axes = plt.subplots(len(test_images), 4, figsize=(18, 5 * len(test_images)))
if len(test_images) == 1:
    axes = np.expand_dims(axes, 0)

titles = ["1. Original Input", "2. Predicted Mask", "3. Cutout (Transparent)", "4. Studio White BG"]

for i, img_path in enumerate(test_images):
    base = os.path.splitext(os.path.basename(img_path))[0]
    
    orig = Image.open(img_path).convert("RGB")
    mask = Image.open(f"{pred_output_dir}/{base}_mask.png")
    nobg = Image.open(f"{pred_output_dir}/{base}_nobg.png")
    white_bg = Image.open(f"{pred_output_dir}/{base}_colored_bg.jpg")
    
    imgs = [orig, mask, nobg, white_bg]
    for j, (im, title) in enumerate(zip(imgs, titles)):
        axes[i, j].imshow(im)
        if i == 0:
            axes[i, j].set_title(title, fontsize=13, fontweight="bold")
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

## 9. Step 5: Package & Download All Artifacts
Compress all trained weights (`best.pth`), training curves, evaluation metrics, and sample predictions into a single downloadable zip file.

In [ ]:
import shutil
from IPython.display import FileLink

zip_path = "/kaggle/working/bg_removal_artifacts.zip"
print(f"Compressing outputs into {zip_path}...")
!zip -rq {zip_path} /kaggle/working/outputs

print("Ready for download!")
display(FileLink("bg_removal_artifacts.zip"))